Author: **Dhrubajyoti Ghosh**

Assignment: Capstone Project 3

Version: 1.0

Question 7: Personal Finance AI Agent (LangChain)

Date: 18 Oct 2026

--------------------------------------------

Personal Finance AI Agent (LangChain)
1. Objective
Build an AI-powered Personal Finance Assistant using LangChain Agents and Tools.
The agent should help users perform financial calculations and planning such as EMI calculation,investment returns, and budget allocation. The LLM must decide which tool to use depending on the user query.
2. Learning Outcomes
By completing this assignment, you will learn:
• How to build AI agents using LangChain
• How to create and integrate custom tools
• How an LLM selects tools based on user queries
• Designing practical AI assistants for real-world tasks
3. Project Description
You will build a Personal Finance AI Agent capable of answering queries like:

• If I invest ₹5000 per month for 10 years at 12% return, how much money will I have?
• Calculate EMI for a ₹10 lakh loan at 9% interest for 20 years.
• How much should I save monthly to reach ₹20 lakh in 10 years?
• Create a monthly budget for someone earning ₹80,000.

The system should use tools to perform calculations instead of relying only on the LLM.
4. Required Tools
Students must implement following tools:

1. Calculator Tool
Used for general mathematical calculations.

2. EMI Calculator Tool
Used to calculate loan EMI.

EMI Formula:
EMI = P × r × (1+r)^n / ((1+r)^n - 1)

Where:
P = Loan Amount
r = Monthly Interest Rate
n = Number of Months

3. SIP Investment Calculator
Calculates future value of monthly investments.

Future Value Formula:
FV = P × ((1+r)^n − 1) / r

Where:
P = Monthly Investment
r = Monthly Return
n = Total Months

4. Budget Planner Tool
Suggest budget allocation using the 50-30-20 rule:
• 50% Needs
• 30% Wants
• 20% Savings
5. Technical Requirements
You must:
• Use LangChain
• Use an LLM (OpenAI / GPT-4o-mini)
• Create at least 3 tools
• Build an Agent that automatically selects tools
• Demonstrate the system using multiple queries
6. Example Queries for Testing
Students must demonstrate at least FIVE queries and Streamlit deployment

Examples:
1. Calculate EMI for ₹10 lakh loan at 8% interest for 20 years
2. If I invest ₹7000 monthly for 15 years at 10% return
3. Create a budget plan for income ₹90,000
4. What is 20% of ₹85,000
5. How much will I get if I invest ₹3000 per month for 5 years at 8% return



------------------
-----------------

In [1]:
#importing Libraries
import time
from datetime import datetime


In [2]:
#importing llm.py
#from llm import llm
#Not required as it is imported from agents.py

In [3]:
#Testing Ollama model. Please note I am using Ollama for development and testing.
#During submission will change it to GPT model
#resp = llm.invoke("what is 2+2?")
#print(resp.content)

In [2]:
from agent import create_finance_ai_agent
#from llm import finance_llm

In [3]:
#instantiating an agent
fi_agent = create_finance_ai_agent()

In [4]:
#Function to test if tool was used for calculation
def debug_agent_response(response):
    print("\n" + "=" * 60)
    print("AGENT EXECUTION TRACE")
    print("=" * 60)

    for i, message in enumerate(response["messages"]):

        print(f"\n--- Message {i} ---")
        print("Type:", type(message).__name__)

        if hasattr(message, "tool_calls") and message.tool_calls:
            print("Tool Calls:")

            for tool_call in message.tool_calls:
                print(f"  Tool: {tool_call['name']}")
                print(f"  Args: {tool_call['args']}")

        if type(message).__name__ == "ToolMessage":
            print("TOOL RESULT:")
            print(message.content)

        elif message.content:
            print("Content:")
            print(message.content)

    print("=" * 60)

---------------
**Using Ollama Models "gemma4:e2b" for testing**

---------------

In [13]:
#Testing debug_agent_response - Question 1: Calculate EMI for ₹10 lakh loan at 8% interest for 20 years
response_1_ollama = fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Calculate EMI for ₹10 lakh loan at 8% interest for 20 years"
        }
    ]
})

debug_agent_response(response_1_ollama)



AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
Calculate EMI for ₹10 lakh loan at 8% interest for 20 years

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: emi_calculator_tool
  Args: {'interest_rate': 8, 'interest_rate_period': 'annual', 'loan_amount': 1000000, 'tenure_months': 240}

--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"monthly_emi": 8364.4, "loan_amount": 1000000.0, "interest_rate": 8.0, "interest_rate_period": "annual", "tenure_months": 240, "total_payment": 2007456.17, "total_interest": 1007456.17}

--- Message 3 ---
Type: AIMessage
Content:
The Equated Monthly Installment (EMI) for a loan of ₹10,00,000 at an 8% annual interest rate for a tenure of 20 years is ₹8,364.40 per month. The total payment over the loan tenure will be ₹20,07,456.17, and the total interest paid will be ₹10,07,456.17.


In [11]:
#Testing debug_agent_response - Question 2: If I invest ₹7000 monthly for 15 years at 10% return
response_2_ollama = fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "If I invest ₹7000 monthly for 15 years at 10% return"
        }
    ]
})

debug_agent_response(response_2_ollama)


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
If I invest ₹7000 monthly for 15 years at 10% return

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: sip_calculator_tool
  Args: {'interest_rate': 10, 'interest_rate_period': 'annual', 'monthly_investment': 7000, 'tenure_months': 180}

--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"monthly_investment": 7000.0, "interest_rate": 10.0, "interest_rate_period": "annual", "tenure_months": 180, "total_investment": 1260000.0, "estimated_returns": 1641292.42, "future_value": 2901292.42}

--- Message 3 ---
Type: AIMessage
Content:
The future value of your investment after 15 years with a monthly investment of ₹7000 at a 10% annual return is ₹2,901,292.42. This includes a total investment of ₹1,260,000.00 and estimated returns of ₹1,641,292.42.


In [9]:
#Testing debug_agent_response - Question 3: Create a budget plan for income ₹90,000
response_3_ollama = fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Create a budget plan for income ₹90,000"
        }
    ]
})

debug_agent_response(response_3_ollama)


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
Create a budget plan for income ₹90,000

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: budget_planner_tool
  Args: {'monthly_income': 90000}
Content:
Based on your monthly income of ₹90,000, here is the budget plan allocated according to the 50-30-20 rule.



--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"monthly_income": 90000.0, "needs": {"percentage": 50, "amount": 45000.0}, "wants": {"percentage": 30, "amount": 27000.0}, "savings": {"percentage": 20, "amount": 18000.0}, "total_allocated": 90000.0}

--- Message 3 ---
Type: AIMessage
Content:
Based on your monthly income of ₹90,000, here is the budget plan allocated according to the 50-30-20 rule.

Needs: ₹45,000.00 (50%)
Wants: ₹27,000.00 (30%)
Savings: ₹18,000.00 (20%)
Total Allocated: ₹90,000.00


In [8]:
#Testing debug_agent_response - Question 4: What is 20% of ₹85,000
response_4_ollama = fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 20% of ₹85,000"
        }
    ]
})

debug_agent_response(response_4_ollama)

Python REPL can execute arbitrary code. Use with caution.



AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
What is 20% of ₹85,000

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: calculator_tool
  Args: {'expression': '0.20 * 85000'}

--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"expression": "0.20 * 85000", "result": "17000.0"}

--- Message 3 ---
Type: AIMessage
Content:
The result of calculating 20% of ₹85,000 is ₹17,000.00.


In [7]:
#Testing debug_agent_response - Question 5: How much will I get if I invest ₹3000 per month for 5 years at 8% return
response_5_ollama = fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How much will I get if I invest ₹3000 per month for 5 years at 8% return"
        }
    ]
})

debug_agent_response(response_5_ollama)


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
How much will I get if I invest ₹3000 per month for 5 years at 8% return

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: sip_calculator_tool
  Args: {'interest_rate': 8, 'interest_rate_period': 'annual', 'monthly_investment': 3000, 'tenure_months': 60}

--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"monthly_investment": 3000.0, "interest_rate": 8.0, "interest_rate_period": "annual", "tenure_months": 60, "total_investment": 180000.0, "estimated_returns": 40430.57, "future_value": 220430.57}

--- Message 3 ---
Type: AIMessage
Content:
Based on your investment details, here is the calculation:

You will invest a total of ₹180,000 over 5 years. With an 8% annual return, the estimated returns on your investment will be ₹40,430.57, resulting in a future value of ₹220,430.57.


In [6]:
#Testing debug_agent_response - Question 6: Out of scope non-mathematical question
response_6_ollama = fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the capital of India?"
        }
    ]
})

debug_agent_response(response_6_ollama)


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
What is the capital of India?

--- Message 1 ---
Type: AIMessage
Content:
I am a Smart Personal Finance AI Assistant and I can only perform mathematical calculations related to finance. I cannot answer general knowledge questions.


---------------
**Using Gemini model for testing**

---------------

In [2]:
from agent import create_finance_gemini_agent

In [3]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [4]:
#instantiating an agent
gemini_fi_agent = create_finance_gemini_agent()

In [5]:
#Function to test if tool was used for calculation for Gemini model
#This function is similar to debug_agent_response but it is for Gemini model which has different response structure
#It is tweaked to display only output text message in content in AI message and exclude extras having signature
def debug_gemini_agent_response(response):
    print("\n" + "=" * 60)
    print("AGENT EXECUTION TRACE")
    print("=" * 60)

    for i, message in enumerate(response["messages"]):

        print(f"\n--- Message {i} ---")
        print("Type:", type(message).__name__)

        if hasattr(message, "tool_calls") and message.tool_calls:
            print("Tool Calls:")

            for tool_call in message.tool_calls:
                print(f"  Tool: {tool_call['name']}")
                print(f"  Args: {tool_call['args']}")

        if type(message).__name__ == "ToolMessage":
            print("TOOL RESULT:")
            print(message.content)

        elif type(message).__name__ == "AIMessage":
            print("Content:")

            if isinstance(message.content, list):
                for block in message.content:
                    if isinstance(block, dict) and block.get("type") == "text":
                        print(block.get("text", ""), end="")

                print()
            else:
                print(message.content)

        elif message.content:
            print("Content:")
            print(message.content)

    print("=" * 60)



In [7]:
#Testing debug_agent_response - Question 1: Calculate EMI for ₹10 lakh loan at 8% interest for 20 years

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_1_gemini = gemini_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Calculate EMI for ₹10 lakh loan at 8% interest for 20 years"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf



In [8]:
#Displaying the response for Question 1
debug_gemini_agent_response(response_1_gemini)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
Calculate EMI for ₹10 lakh loan at 8% interest for 20 years

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: emi_calculator_tool
  Args: {'loan_amount': 1000000, 'interest_rate_period': 'annual', 'interest_rate': 8, 'tenure_months': 240}
Content:


--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"monthly_emi": 8364.4, "loan_amount": 1000000.0, "interest_rate": 8.0, "interest_rate_period": "annual", "tenure_months": 240, "total_payment": 2007456.17, "total_interest": 1007456.17}

--- Message 3 ---
Type: AIMessage
Content:
Interest rate is considered anually.

Here is the EMI calculation for your loan:

- Monthly EMI: ₹8364.40
- Total Interest: ₹1007456.17
- Total Payment: ₹2007456.17

Summary:
For a loan principal of ₹10,00,000 at an annual interest rate of 8% over a tenure of 240 months (20 years), your monthly EMI will be ₹8364.40. The total interest payable over the entire period is ₹10,07,456.17, leading t

In [9]:
#Testing debug_agent_response - Question 2: If I invest ₹7000 monthly for 15 years at 10% return

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_2_gemini = gemini_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "If I invest ₹7000 monthly for 15 years at 10% return, how much will I have at the end?"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf



In [10]:
#Displaying the response for Question 2
debug_gemini_agent_response(response_2_gemini)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
If I invest ₹7000 monthly for 15 years at 10% return, how much will I have at the end?

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: sip_calculator_tool
  Args: {'monthly_investment': 7000, 'tenure_months': 180, 'interest_rate': 10, 'interest_rate_period': 'annual'}
Content:


--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"monthly_investment": 7000.0, "interest_rate": 10.0, "interest_rate_period": "annual", "tenure_months": 180, "total_investment": 1260000.0, "estimated_returns": 1641292.42, "future_value": 2901292.42}

--- Message 3 ---
Type: AIMessage
Content:
Interest rate is considered anually.

Here is the breakdown of your investment:
- Total Investment: ₹1,260,000
- Estimated Returns: ₹1,641,292.42
- Future Value: ₹2,901,292.42

By investing ₹7,000 monthly for 15 years (180 months) at an annual expected return of 10%, your total accumulated corpus will be ₹2,901,292.42.
Execution Finished At: 2026

In [ ]:
#Testing debug_agent_response - Question 3: Create a budget plan for income ₹90,000

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_3_gemini = gemini_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Create a budget plan for income ₹90,000"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf



In [12]:
#Displaying the response for Question 3
debug_gemini_agent_response(response_3_gemini)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
Create a budget plan for income ₹90,000

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: budget_planner_tool
  Args: {'monthly_income': 90000}
Content:


--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"monthly_income": 90000.0, "needs": {"percentage": 50, "amount": 45000.0}, "wants": {"percentage": 30, "amount": 27000.0}, "savings": {"percentage": 20, "amount": 18000.0}, "total_allocated": 90000.0}

--- Message 3 ---
Type: AIMessage
Content:
Here is your monthly budget allocation based on the 50-30-20 rule for a monthly income of ₹90,000:

- Needs (50%): ₹45,000
- Wants (30%): ₹27,000
- Savings (20%): ₹18,000
- Total Allocated: ₹90,000

Under this plan, ₹45,000 is dedicated to essential expenses such as rent, utilities, groceries, and debt payments. ₹27,000 is allocated for personal discretionary spending and lifestyle choices. The remaining ₹18,000 is directed toward emergency funds, investments, and long-t

In [13]:
#Testing debug_agent_response - Question 4: What is 20% of ₹85,000

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_4_gemini = gemini_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 20% of ₹85,000"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf



GoogleRateLimitError: Error calling model 'gemini-3.7-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.7-flash\nPlease retry in 46.031179273s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '

In [6]:
#Testing debug_agent_response - Question 4: What is 20% of ₹85,000

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_4_gemini = gemini_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 20% of ₹85,000"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

GoogleInvalidRequestError: Error calling model 'Gemini 3.1 Flash-Lite' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': '* GenerateContentRequest.model: unexpected model name format\n', 'status': 'INVALID_ARGUMENT'}}

In [ ]:
#Displaying the response for Question 4
debug_gemini_agent_response(response_4_gemini)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")

In [ ]:
#Testing debug_agent_response - Question 5: How much will I get if I invest ₹3000 per month for 5 years at 8% return

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_5_gemini = gemini_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How much will I get if I invest ₹3000 per month for 5 years at 8% return"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf



In [ ]:
#Displaying the response for Question 5
debug_gemini_agent_response(response_5_gemini)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")

In [ ]:
#Testing debug_agent_response - Question 6: What is the capital of India?

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_6_gemini = gemini_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the capital of India?"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

GoogleInvalidRequestError: Error calling model 'Gemini 3.1 Flash-Lite' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': '* GenerateContentRequest.model: unexpected model name format\n', 'status': 'INVALID_ARGUMENT'}}

In [7]:
#Displaying the response for Question 6
debug_gemini_agent_response(response_6_gemini)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")

NameError: name 'response_6_gemini' is not defined

---------------
**Using Ollama model for testing** - llama3.1:latest

---------------

In [1]:
#importing Libraries
import time
from datetime import datetime

In [2]:
#Function to test if tool was used for calculation
def debug_agent_response(response):
    print("\n" + "=" * 60)
    print("AGENT EXECUTION TRACE")
    print("=" * 60)

    for i, message in enumerate(response["messages"]):

        print(f"\n--- Message {i} ---")
        print("Type:", type(message).__name__)

        if hasattr(message, "tool_calls") and message.tool_calls:
            print("Tool Calls:")

            for tool_call in message.tool_calls:
                print(f"  Tool: {tool_call['name']}")
                print(f"  Args: {tool_call['args']}")

        if type(message).__name__ == "ToolMessage":
            print("TOOL RESULT:")
            print(message.content)

        elif message.content:
            print("Content:")
            print(message.content)

    print("=" * 60)

In [3]:
from agent import create_finance_llama_ai_agent
#from llm import finance_llm

In [4]:
#instantiating an agent
llama_fi_agent = create_finance_llama_ai_agent()

In [5]:
#Testing debug_agent_response - Question 1: Calculate EMI for ₹10 lakh loan at 8% interest for 20 years

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_1_llama = llama_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Calculate EMI for ₹10 lakh loan at 8% interest for 20 years"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

In [6]:
#Displaying the response for Question 1
debug_agent_response(response_1_llama)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
Calculate EMI for ₹10 lakh loan at 8% interest for 20 years

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: emi_calculator_tool
  Args: {'interest_rate_period': 'annual', 'interest_rate': '8', 'tenure_months': '240', 'loan_amount': '1000000'}

--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"monthly_emi": 8364.4, "loan_amount": 1000000.0, "interest_rate": 8.0, "interest_rate_period": "annual", "tenure_months": 240, "total_payment": 2007456.17, "total_interest": 1007456.17}

--- Message 3 ---
Type: AIMessage
Content:
The EMI for a ₹10 lakh loan at 8% interest for 20 years is ₹8,364.4. The total amount paid over 20 years will be ₹20,07,456.17, with ₹10,07,456.17 being the total interest paid.
Execution Finished At: 2026-10-02 10:42:56.374
Total Execution Duration: 180.5867 seconds


In [7]:
#Testing debug_agent_response - Question 2: If I invest ₹7000 monthly for 15 years at 10% return

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_2_llama = llama_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "If I invest ₹7000 monthly for 15 years at 10% return"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

In [8]:
#Displaying the response for Question 2
debug_agent_response(response_2_llama)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
If I invest ₹7000 monthly for 15 years at 10% return

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: sip_calculator_tool
  Args: {'monthly_investment': '7000', 'interest_rate': '10', 'tenure_months': '180', 'interest_rate_period': 'monthly'}

--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"monthly_investment": 7000.0, "interest_rate": 10.0, "interest_rate_period": "monthly", "tenure_months": 180, "total_investment": 1260000.0, "estimated_returns": 1975973319180.8, "future_value": 1975974579180.8}

--- Message 3 ---
Type: AIMessage
Content:
After 15 years, your investment of ₹7000 per month at a 10% return is estimated to be worth ₹19,75,97,45,780.80.
Execution Finished At: 2026-10-02 10:46:19.841
Total Execution Duration: 103.4676 seconds


In [9]:
#Testing debug_agent_response - Question 3: Create a budget plan for income ₹90,000

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_3_llama = llama_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Create a budget plan for income ₹90,000"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

In [10]:
#Displaying the response for Question 3
debug_agent_response(response_3_llama)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
Create a budget plan for income ₹90,000

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: budget_planner_tool
  Args: {'monthly_income': '90000'}

--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"monthly_income": 90000.0, "needs": {"percentage": 50, "amount": 45000.0}, "wants": {"percentage": 30, "amount": 27000.0}, "savings": {"percentage": 20, "amount": 18000.0}, "total_allocated": 90000.0}

--- Message 3 ---
Type: AIMessage
Content:
Based on your monthly income of ₹90,000, here's a suggested budget plan:

* Needs: ₹45,000 (50% of income) - essential expenses like rent, utilities, groceries, transportation, and minimum payments on debts.
* Wants: ₹27,000 (30% of income) - discretionary expenses like entertainment, hobbies, travel, and lifestyle upgrades.
* Savings: ₹18,000 (20% of income) - for building an emergency fund, paying off high-interest debts, and long-term investments.

Remember, this is just a st

In [11]:
#Testing debug_agent_response - Question 4: What is 20% of ₹85,000

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_4_llama = llama_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 20% of ₹85,000?"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

Python REPL can execute arbitrary code. Use with caution.


In [12]:
#Displaying the response for Question 4
debug_agent_response(response_4_llama)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
What is 20% of ₹85,000?

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: calculator_tool
  Args: {'expression': '0.2 * 85000'}

--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"expression": "0.2 * 85000", "result": "17000.0"}

--- Message 3 ---
Type: AIMessage
Content:
The result is: ₹17,000.
Execution Finished At: 2026-10-02 10:50:55.039
Total Execution Duration: 79.3374 seconds


In [13]:
#Testing debug_agent_response - Question 5: Calculate EMI for ₹10 lakh loan at 8% interest for 20 years

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_5_llama = llama_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How much will I get if I invest ₹3000 per month for 5 years at 8% return"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

In [14]:
#Displaying the response for Question 5
debug_agent_response(response_5_llama)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
How much will I get if I invest ₹3000 per month for 5 years at 8% return

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: sip_calculator_tool
  Args: {'monthly_investment': '3000', 'interest_rate': '8', 'tenure_months': '60', 'interest_rate_period': 'monthly'}

--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"monthly_investment": 3000.0, "interest_rate": 8.0, "interest_rate_period": "monthly", "tenure_months": 60, "total_investment": 180000.0, "estimated_returns": 3579639.89, "future_value": 3759639.89}

--- Message 3 ---
Type: AIMessage
Content:
If you invest ₹3000 per month for 5 years at an 8% return, you can expect a future value of ₹3,759,639.89.
Execution Finished At: 2026-10-02 10:52:44.297
Total Execution Duration: 99.0369 seconds


In [5]:
#Testing debug_agent_response - Question 6: What is the capital of India?

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_6_llama = llama_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the capital of India?"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

In [6]:
#Displaying the response for Question 6
debug_agent_response(response_6_llama)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
What is the capital of India?

--- Message 1 ---
Type: AIMessage
Content:
I'm not able to answer general knowledge questions like the capital of India.
Execution Finished At: 2026-10-02 11:07:26.203
Total Execution Duration: 127.3529 seconds


In [5]:
#Testing debug_agent_response - Question 7: What is the 5!?

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_7_llama = llama_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the 5 factorial?"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

Python REPL can execute arbitrary code. Use with caution.


In [6]:
#Displaying the response for Question 6
debug_agent_response(response_7_llama)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 0 ---
Type: HumanMessage
Content:
What is the 5 factorial?

--- Message 1 ---
Type: AIMessage
Tool Calls:
  Tool: calculator_tool
  Args: {'expression': 'math.factorial(5)'}

--- Message 2 ---
Type: ToolMessage
TOOL RESULT:
{"expression": "math.factorial(5)", "result": "120"}

--- Message 3 ---
Type: AIMessage
Content:
The result of 5 factorial is 120.
Execution Finished At: 2026-10-02 14:21:09.149
Total Execution Duration: 102.4526 seconds


-----------------------------
**Using GPT Model

-----------------------------

In [1]:
#importing Libraries
import time
from datetime import datetime

In [2]:
#Function to test if tool was used for calculation for Gemini model
#This function kept separate from debug_agent_response as Gemini model has different response structure
def debug_gpt_agent_response(response):
    print("\n" + "=" * 60)
    print("AGENT EXECUTION TRACE")
    print("=" * 60)

    messages = response.get("messages", [])

    for i, message in enumerate(messages, start=1):
        message_type = type(message).__name__

        print(f"\n--- Message {i}: {message_type} ---")

        # AI MESSAGE

        if message_type == "AIMessage":
            content = message.content

            # GPT/LangChain may return content as a list
            # containing text, tool calls, signatures, etc.
            if isinstance(content, list):
                text_parts = []

                for block in content:
                    if isinstance(block, dict):
                        if block.get("type") == "text":
                            text = block.get("text", "")
                            if text:
                                text_parts.append(text)

                if text_parts:
                    print("AI:")
                    print("".join(text_parts))

            # Simple string content
            elif isinstance(content, str):
                print("AI:")
                print(content)

            # Show tools requested by the AI
            tool_calls = getattr(message, "tool_calls", None)

            if tool_calls:
                print("Tools used:")

                for tool_call in tool_calls:
                    tool_name = tool_call.get("name", "Unknown tool")
                    args = tool_call.get("args", {})

                    print(f"  - {tool_name}")
                    if args:
                        print(f"    Args: {args}")

        # TOOL MESSAGE
        elif message_type == "ToolMessage":
            tool_name = getattr(message, "name", None)

            print("Tool:")
            if tool_name:
                print(f"  Name: {tool_name}")

            print("  Result:")
            print(f"  {message.content}")

        # OTHER MESSAGE TYPES
        else:
            content = getattr(message, "content", None)

            if content:
                print("Content:")
                print(content)

    print("\n" + "=" * 60)

In [3]:
from agent import create_gpt_finance_ai_agent
#from llm import finance_llm

In [4]:
#instantiating an agent
gpt_fi_agent = create_gpt_finance_ai_agent()

In [6]:
#Testing debug_agent_response - Question 1: Calculate EMI for ₹10 lakh loan at 8% interest for 20 years

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_1_gpt = gpt_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Calculate EMI for ₹10 lakh loan at 8% interest for 20 years"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()

In [8]:
#Displaying the response for Question 1
debug_gpt_agent_response(response_1_gpt)

duration = end_perf - start_perf
print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")



AGENT EXECUTION TRACE

--- Message 1: HumanMessage ---
Content:
Calculate EMI for ₹10 lakh loan at 8% interest for 20 years

--- Message 2: AIMessage ---
AI:

Tools used:
  - emi_calculator_tool
    Args: {'loan_amount': 1000000, 'interest_rate': 8, 'tenure_months': 240}

--- Message 3: ToolMessage ---
Tool:
  Name: emi_calculator_tool
  Result:
  {"monthly_emi": 8364.4, "loan_amount": 1000000.0, "interest_rate": 8.0, "interest_rate_period": "annual", "tenure_months": 240, "total_payment": 2007456.17, "total_interest": 1007456.17}

--- Message 4: AIMessage ---
AI:
Your EMI for a ₹10 lakh loan at 8% interest for 20 years is ₹8,364.40. 

Total payment over the tenure will be ₹20,07,456.17, with total interest amounting to ₹10,07,456.17.

Execution Finished At: 2026-10-02 15:53:45.119
Total Execution Duration: 4.1078 seconds


In [9]:
#Testing debug_agent_response - Question 2: If I invest ₹7000 monthly for 15 years at 10% return, how much will I have at the end?

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_2_gpt = gpt_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "If I invest ₹7000 monthly for 15 years at 10% return, how much will I have at the end?"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

In [10]:
#Displaying the response for Question 2
debug_gpt_agent_response(response_2_gpt)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 1: HumanMessage ---
Content:
If I invest ₹7000 monthly for 15 years at 10% return, how much will I have at the end?

--- Message 2: AIMessage ---
AI:

Tools used:
  - sip_calculator_tool
    Args: {'monthly_investment': 7000, 'interest_rate': 10, 'tenure_months': 180}

--- Message 3: ToolMessage ---
Tool:
  Name: sip_calculator_tool
  Result:
  {"monthly_investment": 7000.0, "interest_rate": 10.0, "interest_rate_period": "annual", "tenure_months": 180, "total_investment": 1260000.0, "estimated_returns": 1641292.42, "future_value": 2901292.42}

--- Message 4: AIMessage ---
AI:
You will have approximately ₹2,901,292.42 at the end of 15 years.

Execution Finished At: 2026-10-02 15:54:42.846
Total Execution Duration: 4.8559 seconds


In [11]:
#Testing debug_agent_response - Question 3: Create a budget plan for income ₹90,000

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_3_gpt = gpt_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Create a budget plan for income ₹90,000"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

In [12]:
#Displaying the response for Question 3
debug_gpt_agent_response(response_3_gpt)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 1: HumanMessage ---
Content:
Create a budget plan for income ₹90,000

--- Message 2: AIMessage ---
AI:

Tools used:
  - budget_planner_tool
    Args: {'monthly_income': 90000}

--- Message 3: ToolMessage ---
Tool:
  Name: budget_planner_tool
  Result:
  {"monthly_income": 90000.0, "needs": {"percentage": 50, "amount": 45000.0}, "wants": {"percentage": 30, "amount": 27000.0}, "savings": {"percentage": 20, "amount": 18000.0}, "total_allocated": 90000.0}

--- Message 4: AIMessage ---
AI:
For a monthly income of ₹90,000, the budget plan is as follows:

- Needs: ₹45,000 (50%)
- Wants: ₹27,000 (30%)
- Savings: ₹18,000 (20%)

Total allocated: ₹90,000.

Execution Finished At: 2026-10-02 15:55:00.513
Total Execution Duration: 3.0135 seconds


In [13]:
#Testing debug_agent_response - Question 4: What is 20% of ₹85,000

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_4_gpt = gpt_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 20% of ₹85,000?"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

Python REPL can execute arbitrary code. Use with caution.


In [14]:
#Displaying the response for Question 4
debug_gpt_agent_response(response_4_gpt)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 1: HumanMessage ---
Content:
What is 20% of ₹85,000?

--- Message 2: AIMessage ---
AI:

Tools used:
  - calculator_tool
    Args: {'expression': '0.2 * 85000'}

--- Message 3: ToolMessage ---
Tool:
  Name: calculator_tool
  Result:
  {"expression": "0.2 * 85000", "result": "17000.0"}

--- Message 4: AIMessage ---
AI:
20% of ₹85,000 is ₹17,000.

Execution Finished At: 2026-10-02 15:55:43.939
Total Execution Duration: 3.2978 seconds


In [5]:
#Testing debug_agent_response - Question 5: How much will I get if I invest ₹3000 per month for 5 years at 8% return

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_5_gpt = gpt_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How much will I get if I invest ₹3000 per month for 5 years at 8% return"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

In [6]:
#Displaying the response for Question 5
debug_gpt_agent_response(response_5_gpt)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 1: HumanMessage ---
Content:
How much will I get if I invest ₹3000 per month for 5 years at 8% return

--- Message 2: AIMessage ---
AI:

Tools used:
  - sip_calculator_tool
    Args: {'monthly_investment': 3000, 'interest_rate': 8, 'tenure_months': 60}

--- Message 3: ToolMessage ---
Tool:
  Name: sip_calculator_tool
  Result:
  {"monthly_investment": 3000.0, "interest_rate": 8.0, "interest_rate_period": "annual", "tenure_months": 60, "total_investment": 180000.0, "estimated_returns": 40430.57, "future_value": 220430.57}

--- Message 4: AIMessage ---
AI:
If you invest ₹3000 per month for 5 years at an 8% return, you will receive approximately ₹220430.57. Your total investment will be ₹180000, with estimated returns of ₹40430.57.

Execution Finished At: 2026-10-02 16:04:28.439
Total Execution Duration: 2.7120 seconds


In [17]:
#Testing debug_agent_response - Question 6: What is the capital of India?

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_6_gpt = gpt_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the capital of India?"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

In [18]:
#Displaying the response for Question 6
debug_gpt_agent_response(response_6_gpt)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 1: HumanMessage ---
Content:
What is the capital of India?

--- Message 2: AIMessage ---
AI:
This is beyond my knowledge please try asking any other question.

Execution Finished At: 2026-10-02 15:56:01.878
Total Execution Duration: 1.1999 seconds


In [19]:
#Testing debug_agent_response - Question 7: What is the factorial of 5?

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_7_gpt = gpt_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is the factorial of 5?"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

In [20]:
#Displaying the response for Question 7
debug_gpt_agent_response(response_7_gpt)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 1: HumanMessage ---
Content:
What is the factorial of 5?

--- Message 2: AIMessage ---
AI:

Tools used:
  - calculator_tool
    Args: {'expression': 'math.factorial(5)'}

--- Message 3: ToolMessage ---
Tool:
  Name: calculator_tool
  Result:
  {"expression": "math.factorial(5)", "result": "120"}

--- Message 4: AIMessage ---
AI:
The factorial of 5 is 120.

Execution Finished At: 2026-10-02 15:57:04.856
Total Execution Duration: 2.6297 seconds


In [21]:
#Testing debug_agent_response - Question 8: What is sine 30 degrees?

start_time_dt = datetime.now()
start_perf = time.perf_counter()

response_8_gpt = gpt_fi_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is sine 30 degrees?"
        }
    ]
})
end_time_dt = datetime.now()
end_perf = time.perf_counter()
duration = end_perf - start_perf

In [22]:
#Displaying the response for Question 8
debug_gpt_agent_response(response_8_gpt)


print(f"Execution Finished At: {end_time_dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]}")
print(f"Total Execution Duration: {duration:.4f} seconds")


AGENT EXECUTION TRACE

--- Message 1: HumanMessage ---
Content:
What is sine 30 degrees?

--- Message 2: AIMessage ---
AI:

Tools used:
  - calculator_tool
    Args: {'expression': 'math.sin(math.radians(30))'}

--- Message 3: ToolMessage ---
Tool:
  Name: calculator_tool
  Result:
  {"expression": "math.sin(math.radians(30))", "result": "0.49999999999999994"}

--- Message 4: AIMessage ---
AI:
The sine of 30 degrees is approximately 0.5.

Execution Finished At: 2026-10-02 15:57:49.875
Total Execution Duration: 2.4690 seconds


-Being paid, response from GPT toll is the fastest. 

-Whereas speed of "gemini-3.7-flash" seemed comparable to "gemma4:e2b". 

-"gemma4:e2b" is locally installed and Gemini API usede to access  "gemini-3.7-flash". 

-"llama3.1:latest" was the slowest to respond.

-"gemini-3.8-flash" free version had better response time (almost half of "gemini-3.7-flash") however it had very limited usage availability.

-One challenge observed with "gemma4:e2b" in 1 out of 5 times, it provided AI message instead of using tool. On re-run, it utilised the tool back again.
